# Complaint Triage System: isi database demo dengan prediksi IndoBERT

Menilai 52 ribu ulasan dengan IndoBERT di CPU butuh sekitar 2 jam; di GPU Colab hanya beberapa menit.
Notebook ini memuat ulasan + prediksi IndoBERT ke database PostgreSQL (Neon) yang dipakai demo.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Siapkan: connection string Neon, nama model di Hugging Face Hub, dan file `data/annotated/brimo.jsonl`.
3. Jalankan sel dari atas ke bawah.

In [ ]:
# 1) Kode dan library (sekitar 1 menit). Torch bawaan Colab dipakai apa adanya.
!git clone -q https://github.com/HachiVin/Complaint-Triage-System.git
%cd Complaint-Triage-System
!pip install -q "psycopg[binary,pool]==3.3.6" transformers==5.20.2 tokenizers==0.23.3 scikit-learn==1.9.1 joblib==1.6.0 pydantic==2.14.1 huggingface_hub==2.3.0
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA - ganti runtime ke T4 GPU dulu")

In [ ]:
# 2) Unggah data/annotated/brimo.jsonl dari laptop
import os, shutil
from google.colab import files
os.makedirs("data/annotated", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"data/annotated/{name}")
!ls -la data/annotated

In [ ]:
# 3) Connection string Neon dan nama model. Input disembunyikan dan tidak tersimpan di notebook.
from getpass import getpass
os.environ["DATABASE_URL"] = getpass("Neon connection string (postgresql://...): ").strip()
MODEL = input("Model di Hugging Face Hub (contoh: namakamu/complaint-triage-indobert): ").strip()

In [ ]:
# 4) Muat ulasan + label dataset, lalu nilai dengan IndoBERT (beberapa menit di T4)
!python scripts/ingest.py data/annotated/brimo.jsonl --model hf://{MODEL} --batch-size 512

In [ ]:
# 5) Cek isi database
import psycopg
with psycopg.connect(os.environ["DATABASE_URL"]) as conn:
    print(conn.execute("SELECT count(*), min(review_date), max(review_date) FROM v_reviews").fetchone())
    print(conn.execute("SELECT model_version, count(*) FROM predictions GROUP BY 1").fetchall())
    print(conn.execute("SELECT urgency, count(*) FROM v_reviews GROUP BY 1 ORDER BY 2 DESC").fetchall())